# 1단계 PLATOON1 — 학생 학습·무리 도착 세계 심판 평가·판정

`PLATOON1_data` 세 조각의 64판이 모두 끝난 뒤 **런타임 하나**(CPU + 높은 RAM)에서 **모두 실행**합니다.
1. 날짜 보류 진단(참고, 약 10분): 학습에 안 쓴 날짜에서 '더 나은 녹색'을 얼마나 맞히는지 봅니다. 판정·선택에는 쓰지 않습니다.
2. 학생 학습(약 5분): 같은 자료로 난수 42·43·44 세 신경망을 학습합니다. 입력은 관측 167칸(ARENA3 119칸 + 시간표 관측 48칸)입니다.
3. 심판 평가: 모델 3개 × 평가 84상황 = 252판(백그라운드, CPU 8개 약 3시간). 기준 M4·TM4·OM4·P1M4는 PLATOON0 결과를 씁니다.
4. 판정(사전 등록): PT1이 M4 대비 ARENA3 판정을 통과하고, **학습의 몫 L = PT1 − TM4**가 0.3초 이상·하한 > 0·양수 날짜 5개 이상이면 통과.
5. 확인: 4가 통과했을 때만 확인 84상황에서 M4·TM4·PT1을 한 번 더 잽니다(약 4~5시간). 통과 못 하면 SUMO를 돌리지 않고 끝납니다.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, json

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/FutureTraffic")
ARCHIVE = DRIVE / "FutureTraffic_PLATOON1.zip"
PLATOON0 = DRIVE / "explore_platoon0"          # PLATOON0 결과(무리 도착 세계 M4·TM4·OM4·P1M4, 읽기만)
OUT = DRIVE / "explore_platoon1"
WORKSPACE = Path("/content/FutureTraffic_PLATOON1")
MIN_CPUS = 8
for path in (ARCHIVE, PLATOON0 / "experiment.json", PLATOON0 / "runs" / "P" / "TM4"):
    assert path.exists(), f"필요한 파일이 없습니다: {path}"
CPUS = os.cpu_count() or 1
print("CPU 개수:", CPUS, "| 결과 폴더:", OUT)
if CPUS < MIN_CPUS:
    print(f"주의: CPU가 {CPUS}개입니다. 평가가 오래 걸립니다. 가능하면 CPU + 높은 RAM 런타임을 쓰세요.")
done = len(list((OUT / "data" / "runs" / "PT1_DATA").glob("*/cache.json")))
assert done == 64, f"자료가 64판 중 {done}판만 끝났습니다. PLATOON1_data 세 조각을 먼저 끝내세요."

In [ ]:
# ZIP 전체의 파일 목록·SHA-256을 확인한 뒤 검증된 전용 추출기를 사용합니다.
import hashlib, stat, types, zipfile
from pathlib import PurePosixPath

with zipfile.ZipFile(ARCHIVE) as archive:
    names = archive.namelist()
    assert len(names) == len(set(names)) and names.count("manifest.json") == 1, "ZIP 파일 목록 오류"
    package_manifest = json.loads(archive.read("manifest.json"))
    assert package_manifest.get("kind") == "WEIGHT1_PACKAGE_V1", "검증할 수 없는 ZIP 형식입니다"
    assert package_manifest.get("experiment") == "PLATOON1_V1", "PLATOON1 전용 ZIP이 아닙니다"
    assert package_manifest.get("source_models_included") is False, "모델·결과가 든 묶음은 쓰지 않습니다"
    expected = package_manifest["files"]
    assert expected and "manifest.json" not in expected and set(names) == set(expected) | {"manifest.json"}, "ZIP과 manifest 목록 불일치"
    for info in archive.infolist():
        name, path = info.filename, PurePosixPath(info.filename)
        assert path.parts and not path.is_absolute() and ".." not in path.parts and path.as_posix() == name, "잘못된 ZIP 경로"
        assert "\\" not in name and not info.is_dir() and not stat.S_ISLNK(info.external_attr >> 16), "잘못된 ZIP 파일 형식"
        if name != "manifest.json":
            assert hashlib.sha256(archive.read(name)).hexdigest() == expected[name], f"SHA-256 불일치: {name}"
    source_code = archive.read("src/explore_weight_package.py").decode("utf-8")

bootstrap = types.ModuleType("platoon1_verified_package")
bootstrap.__file__ = str(WORKSPACE / "src/explore_weight_package.py")
exec(compile(source_code, bootstrap.__file__, "exec"), bootstrap.__dict__)
package_manifest = bootstrap.extract_verified(ARCHIVE, WORKSPACE)
print(f"검증·추출 완료: {len(package_manifest['files'])}개 파일 → {WORKSPACE} | 잠금 {package_manifest['lock_sha256'][:12]} "
      f"| PLATOON1 설정 {package_manifest['platoon1_config_sha256'][:12]} | 사전 등록 {package_manifest['prereg_sha256'][:12]}")

In [ ]:
import subprocess, importlib.metadata

os.environ["FOCUSED_RULESET"] = "pet1"
os.environ["FOCUSED_NORTH_EXT"] = "240"
# ARENA3 심판 기준선과 같은 평가 지문이 되도록 같은 패키지 버전을 씁니다.
EXPECTED = {"numpy": "2.1.3", "torch": "2.11.0+cpu", "stable-baselines3": "2.9.0",
            "sumo-rl": "1.4.5", "eclipse-sumo": "1.27.1"}

def installed_versions():
    versions = {}
    for name in EXPECTED:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    return versions

before = installed_versions()
if before == EXPECTED:
    print("이전 실험과 패키지 버전이 같아 재설치를 건너뜁니다.")
else:
    print("이전 실험과 같은 패키지 버전을 준비합니다:", before)
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "numpy==2.1.3", "stable-baselines3==2.9.0", "eclipse-sumo==1.27.1"],
                   cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.11.0+cpu", "--index-url", "https://download.pytorch.org/whl/cpu"],
                   cwd=WORKSPACE, check=True)
after = installed_versions()
assert after == EXPECTED, f"패키지 버전이 이전 실험과 다릅니다(평가 지문이 달라져 기준선과 비교할 수 없어 멈춥니다): {after}"
print("실행 패키지 검증 통과:", after)

In [ ]:
sys.path.insert(0, str(WORKSPACE / "src"))
import arena3_colab as colab
from IPython.display import display, Markdown

for name in ("test_explore_platoon1.py", "test_explore_platoon0.py", "test_arena3.py"):
    colab.run_live(["-m", "unittest", "discover", "-s", "tests", "-p", name], WORKSPACE)
print("실행 전 검사(SUMO 없이) 통과")

In [ ]:
# 1) 날짜 보류 진단(참고, SUMO 없음).
OFFLINE = OUT / "offline"
if not (OFFLINE / "offline_report.md").exists():
    colab.run_live(["src/explore_platoon1.py", "offline", "--data", str(OUT / "data"), "--out", str(OFFLINE)], WORKSPACE)
display(Markdown((OFFLINE / "offline_report.md").read_text()))

In [ ]:
# 2) 학생 학습(난수 42·43·44, SUMO 없음). 이미 있으면 같은 자료로 학습했는지만 확인하고 건너뜁니다.
assert "colab" in globals(), "처음 네 칸(1~4)을 먼저 실행하세요."
colab.run_live(["src/explore_platoon1.py", "train", "--data", str(OUT / "data"), "--out", str(OUT / "students")], WORKSPACE)

In [ ]:
# 3) 심판 평가: 모델 3개 × 84상황 = 252판(백그라운드). 끊겨도 다시 실행하면 끝난 판은 건너뜁니다.
assert "colab" in globals(), "처음 네 칸(1~4)을 먼저 실행하세요. 런타임이 끊기면 변수와 작업 폴더가 사라집니다."
missing = [s for s in (42, 43, 44) if not (OUT / "students" / f"seed_{s}" / "model.pt").exists()]
assert not missing, f"학생 모델이 없습니다(난수 {missing}). 2)를 먼저 실행하세요."
LOGS = OUT / "logs"


def describe(name):
    done = len(list((OUT / "eval" / "PT1" / "runs").glob("PT1_m*/*/cache.json")))
    failed = len(list((OUT / "eval" / "PT1" / "runs").glob("PT1_m*/*.failed.json")))
    return f"끝난 판 {done}/252, 실패 기록 {failed}"


pids = colab.launch({"evaluate_PT1": ["src/explore_platoon1.py", "evaluate", "--students", str(OUT / "students"),
                                      "--out", str(OUT / "eval" / "PT1"), "--work", "/content/pt1_eval",
                                      "--workers", str(CPUS)]}, WORKSPACE, LOGS)
code = colab.watch(pids, LOGS, 600, describe)
assert code == 0, "평가에서 오류가 났습니다. 위 로그를 저에게 보내 주세요(다시 실행하면 끝난 판은 건너뜁니다)."

In [ ]:
# 4) 판정(SUMO 없음, 몇 분): 기준은 PLATOON0의 무리 도착 세계 실행.
assert "colab" in globals(), "처음 네 칸(1~4)을 먼저 실행하세요."
code = colab.run_live(["src/explore_platoon1.py", "judge", "--eval", str(OUT / "eval" / "PT1"), "--platoon0", str(PLATOON0),
                       "--out", str(OUT / "judge")], WORKSPACE, check=False)
assert code == 0, "아직 평가가 끝나지 않았습니다(위 '아직 끝나지 않은 실행'). 평가가 끝난 뒤 다시 실행하세요."
display(Markdown((OUT / "judge" / "summary.md").read_text()))

In [ ]:
# 5) 확인(확인 84상황, 무리 도착 세계): 4)의 주 판정이 통과했을 때만 M4·TM4·PT1을 돌립니다(약 4~5시간, 백그라운드).
assert "colab" in globals(), "처음 네 칸(1~4)을 먼저 실행하세요."
LOGS = OUT / "logs"
pids = colab.launch({"confirm_PT1": ["src/explore_platoon1.py", "confirm", "--judge", str(OUT / "judge"),
                                     "--students", str(OUT / "students"), "--out", str(OUT / "judge" / "confirm"),
                                     "--work", "/content/pt1_confirm", "--workers", str(CPUS)]}, WORKSPACE, LOGS)
code = colab.watch(pids, LOGS, 600)
result = colab.read_json(OUT / "judge" / "confirm" / "confirmation.json")
print("확인 결과:", result and result.get("status"))
if (OUT / "judge" / "confirm" / "summary.md").exists():
    display(Markdown((OUT / "judge" / "confirm" / "summary.md").read_text()))